## Reproduce plots

In [1]:
import sys;
if ".." not in sys.path: sys.path.append("..") # allows us to have visibility on our package without installing it in editing mode
import matplotlib.pyplot as plt
import matplotlib as mpl
from monaqa2.data.hyperparams import load_best_qemc_gamma_t, export_best_gamma_t_h5
from monaqa2.data.instances import load_instances
from monaqa2.data.runtime import tight_schedule_annealing, make_prefix_stable_schedule_generator
from monaqa2.data.spectral_gap import run_experiment_to_generate_spectral_gaps, load_spectral_gap
from monaqa2.data.classical_query import run_experiment_to_generate_classical_queries
from monaqa2.mcmc.model import IsingModel, RandomIsingModel
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

# 1. Base sizes (APS recommendations for single-column figures)
REGULAR_SIZE = 10  # For axis labels and titles
SMALL_SIZE = 8     # For tick labels and legends

mpl.use("pgf")
mpl.rcParams.update({
    # --- PGF & LaTeX Engine Setup ---
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    
    # --- Font Family (APS Sans-Serif Style) ---
    "font.family": "serif",
    "pgf.preamble": "",
    
    # --- EXPLICIT FONT SIZES ---
    "font.size": REGULAR_SIZE,        # Global fallback size
    "axes.labelsize": REGULAR_SIZE,   # Size of X and Y labels (e.g., "Time")
    "axes.titlesize": REGULAR_SIZE,   # Size of the plot title
    "xtick.labelsize": SMALL_SIZE,    # Size of X-axis tick numbers
    "ytick.labelsize": SMALL_SIZE,    # Size of Y-axis tick numbers
    "legend.fontsize": SMALL_SIZE,    # Size of the legend text
    "figure.titlesize": REGULAR_SIZE, # Size of suptitle (if used)
})

## Appendix SK model 

In [9]:
from monaqa2.data.plot_aux import plot_sk_upper_bounds

fig, ax = plt.subplots(figsize=(13 / 2.54, 5.6 / 2.54))
plot_sk_upper_bounds(fig=fig, ax=ax, output_file=None, legend_placement="right")
fig.tight_layout()

filename = "plots/aux/sk_bounds"
fig.savefig(filename + ".png", dpi=300, bbox_inches="tight")
fig.savefig(filename + ".pgf", dpi=300, bbox_inches="tight")
plt.close(fig)

## Spectral gap plots

In [2]:
from monaqa2.data.plot_table import plot_spectral_gap_vs_beta_table
from monaqa2.data.plot_table import plot_spectral_gap_vs_n_table
from monaqa2.data.plot_table import plot_last_step_classical_queries_and_spectral_gap_vs_n_table
from monaqa2.data.plot_table import plot_last_step_classical_queries_and_quantum_queries_vs_n_table
from monaqa2.data.plot_table import plot_annealing_classical_and_quantum_queries_vs_n_table

#### Spectral gap vs beta

In [3]:
plt.close()
fig, ax = plot_spectral_gap_vs_beta_table(
    fixed_ns=[5, 6, 7, 8, 9, 10], 
    miniature_scale=1.25, 
    ncols=3, 
    statistic="mean+std",
    wspace=0.12,
    show_legend=True,
    legend_y=0,
    legend_fontsize=14,
    figsize=(27.0 / 2.54, 15.75 / 2.54),
    inset_bounds=(0.125, 0.100, 0.438, 0.369),
    
)
fig.tight_layout()
filename = f"plots/appx/spectral_gap_vs_beta"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [4]:
plt.close()
fig, ax = plot_spectral_gap_vs_beta_table(
    fixed_ns=[5, 6, 7, 8, 9, 10], 
    ncols=3, 
    statistic="mean+std-tail",
    wspace=0.12,
    show_legend=True,
    legend_y=0,
    legend_fontsize=14,
    figsize=(27.0 / 2.54, 15.75 / 2.54),
    inset_bounds=(0.125, 0.100, 0.438, 0.369)
)
fig.tight_layout()
filename = f"plots/appx/spectral_gap_vs_beta_notail"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

#### Spectral gap vs n

In [11]:
plt.close()
fig, ax = plot_spectral_gap_vs_n_table(
    fixed_betas=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0], 
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std", 
    show_legend=True,
    y_limit_measurement_scale=1.0,
    y_limit_max_extra_orders=5.0,
    y_tick_order_step=3,
    wspace=0.12,
    legend_fontsize=12,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    extrapolation_alpha_factor=0.95,
)
filename = f"plots/appx/spectral_gap_vs_n"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [6]:
plt.close()
fig, ax = plot_spectral_gap_vs_n_table(
    fixed_betas=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0], 
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std-tail",
    y_limit_measurement_scale=1.0,
    y_limit_max_extra_orders=5.0,
    y_tick_order_step=3,
    wspace=0.12,
    legend_fontsize=10,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    last_row_legend_y=-0.18,
    extrapolation_alpha_factor=0.95)
filename = f"plots/appx/spectral_gap_vs_n_notail"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Classical queries plot

In [7]:
plt.close()
fig, ax, ax_gap = plot_last_step_classical_queries_and_spectral_gap_vs_n_table(
    betas=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0], 
    epsilon=1e-2,
    show_inverse_gap=True,
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=4.0,
)
filename = f"plots/appx/classical_queries_eps-2"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [8]:
plt.close()
fig, ax, ax_gap = plot_last_step_classical_queries_and_spectral_gap_vs_n_table(
    betas=[1.0, 2.0, 5.0] * 3, 
    epsilon=[1e-2] * 3 + [1e-4] * 3 + [1e-8] * 3,
    show_inverse_gap=True,
    ncols=3, 
    n_plot_min=1, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=4.0,
)
filename = f"plots/appx/classical_queries_all_epsilon"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Quantum queries plot

In [3]:
plt.close()
fig, ax = plot_last_step_classical_queries_and_quantum_queries_vs_n_table(
    betas=[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0], 
    epsilon=1e-2,
    ncols=3, 
    n_plot_min=4, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=4.0,
)
filename = "plots/appx/classical_and_quantum_queries_eps-2"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [4]:
plt.close()
fig, ax = plot_last_step_classical_queries_and_quantum_queries_vs_n_table(
    betas=[1.0, 2.0, 5.0] * 3, 
    epsilon=[1e-2] * 3 + [1e-4] * 3 + [1e-8] * 3,
    ncols=3, 
    n_plot_min=4, 
    n_plot_max=20, 
    statistic="mean+std",
    legend_x_span=0.64,
    legend_two_column_x_span=0.34,
    legend_line_half_width=0.050,
    wspace=0.12,
    extrapolation_alpha_factor=0.95,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
    last_row_legend_y=-0.18,
    remove_local_beta_threshold=4.0,
)
filename = "plots/appx/classical_and_quantum_queries_all_epsilon"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

### Runtime plots

In [6]:
from monaqa2.data.plot_table_4 import (
    plot_annealing_classical_and_quantum_runtime_vs_n_table,
    plot_uniform_classical_and_qemc_arithmetic_runtime_vs_n_table,
)

betas = [1.0, 2.0, 5.0] * 3
epsilons = [1e-2] * 3 + [1e-4] * 3 + [1e-8] * 3

physical_error_rate_min = 1e-4
physical_error_rate_max = 1e-4
physical_operation_time_min = 200e-9
physical_operation_time_max = 20_000e-9
physical_measurement_time_min = 20e-9
physical_measurement_time_max = 2_000e-9
num_trotter_steps = 50

plt.close()
fig, axs = plot_uniform_classical_and_qemc_arithmetic_runtime_vs_n_table(
    betas=betas,
    epsilon=epsilons,
    annealing_schedule_generator=tight_schedule_annealing,
    ncols=3,
    n_plot_min=3,
    n_plot_max=100,
    n_fit_min=5,
    n_fit_max=10,
    statistic="mean+std",
    classical_device="fpga",
    physical_error_rate_min=physical_error_rate_min,
    physical_error_rate_max=physical_error_rate_max,
    physical_operation_time_min=physical_operation_time_min,
    physical_operation_time_max=physical_operation_time_max,
    physical_measurement_time_min=physical_measurement_time_min,
    physical_measurement_time_max=physical_measurement_time_max,
    num_trotter_steps=num_trotter_steps,
    show_legend=False,
    show_regime_separator=True,
    show_one_year_line=False,
    show_direct_enumeration_line=False,
    optimistic_pessimistic_intercept=False,
    runtime_ymin_seconds=1.0,
    runtime_ymax_years=1000.0,
    line_width=1.8,
    line_alpha=0.94,
    band_alpha=0.18,
    hspace=0.25,
    wspace=0.12,
    figsize=(33.0 / 2.54, 33.0 / 2.54),
)

filename = "plots/appx/runtime_uniform_vs_qemc_arithmetics_fpga"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

# -----

## Plot queries APS


In [2]:
from monaqa2.data.plot_main_queries import plot_annealing_classical_and_quantum_queries_vs_n

In [108]:
plt.close()
fig, axs = plot_annealing_classical_and_quantum_queries_vs_n(
    beta=4, 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing, 
    n_plot_min=3, 
    n_plot_max=50,
    debug=False,
    show_numerical_boundary=True,
    legend_placement="top_left",
    show_legend=True,
    show_zoom_in=False,
    x_right_padding=4.0,
)
filename = f"plots/main_queries/queries_no_miniature"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

In [109]:
plt.close()
fig, axs = plot_annealing_classical_and_quantum_queries_vs_n(
    beta=4, 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing, 
    n_plot_min=3, 
    n_plot_max=50,
    debug=False,
    show_numerical_boundary=True,
    legend_placement="top_left",
    show_legend=True,
    show_zoom_in=True,
    zoom_xlim=(6, 10),
    zoom_ylim=(1e3, 1e4),
    zoom_bbox=(0.10, 0.45, 0.20, 0.22),
    zoom_tick_fontsize=7.0,
    x_right_padding=4.0,
    show_zoom_in_border=True
)
filename = f"plots/main_queries/queries_miniature"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Plot runtime APS

In [2]:
from monaqa2.data.plot_main_runtime13 import plot_annealing_classical_and_quantum_runtime_vs_n, plot_annealing_classical_and_quantum_runtime_vs_n_three

In [3]:
plt.close()
fig, axs = plot_annealing_classical_and_quantum_runtime_vs_n(
    beta=4, 
    epsilon=1e-2,
    annealing_schedule_generator=tight_schedule_annealing, 
    n_plot_min=3, 
    n_plot_max=100,
    debug=False,
    show_regime_separator=True,
    show_one_year_line=True,
    legend_placement="bottom_right",
    optimistic_pessimistic_intercept=False,
    show_direct_enumeration_line=True,
    show_legend=True,
    x_right_padding=4.0,
    y_top_padding=4.0,
)
filename = f"plots/main_runtime/runtime_no_intercept_w_direct_10ns"
plt.savefig(filename + ".png", dpi=300)
plt.savefig(filename + ".pgf", dpi=300)

## Full annealing plots - queries

In [22]:
from monaqa2.data.plot_main21 import plot_annealing_classical_and_quantum_queries_vs_n

In [23]:
beta = 4.0
epsilon = 1e-2
statistic = "mean+std"
MODES = ["full", "compact", "compact_no_layden"]
LEGEND_PLACEMENTS = ["legend_out", "legend_line"]

for mode in MODES:
    for legend_placement in LEGEND_PLACEMENTS:
        filename = f"plots/main_queries/main_classical_quantum_queries_vs_n_beta{beta:g}_eps{epsilon:.0e}_{statistic.replace('+', '_').replace('/', '_')}_{mode}_{legend_placement}"
        plt.close()
        fig, ax = plot_annealing_classical_and_quantum_queries_vs_n(
            beta=beta,
            epsilon=epsilon,
            mode=mode,
            legend_placement=legend_placement,
            annealing_schedule_generator=tight_schedule_annealing,
            statistic=statistic,
            n_plot_min=3,
            n_plot_max=100,
            n_fit_min=5,
            n_fit_max=10,
            debug=False,
            show_schedule_panel=False,
            show_schedule_vertical_lines=False,
            legend_y_shift=-0.14,
            xlabel_labelpad=0
        )
        plt.savefig(filename + ".png", dpi=300)
        plt.savefig(filename + ".pgf", dpi=300)
        print("X", end="")

XXXXXX

# Runtime

In [157]:
# from monaqa2.data.plot_main22 import plot_annealing_classical_and_quantum_runtime_vs_n, plot_annealing_classical_and_quantum_runtime_fancy_vs_n
# 
# beta = 4.0
# epsilon = 1e-4
# statistic = "mean+std"
# MODES = ["compact", "compact_no_layden"]
# LEGEND_PLACEMENTS = ["legend_out"]
# 
# for mode in MODES:
#     for legend_placement in LEGEND_PLACEMENTS:
#         filename = f"plots/main_runtime/main_classical_quantum_runtime_vs_n_beta{beta:g}_eps{epsilon:.0e}_{statistic.replace('+', '_').replace('/', '_')}_{mode}_{legend_placement}"
#         plt.close()
#         fig, ax = plot_annealing_classical_and_quantum_runtime_fancy_vs_n(
#             beta=beta,
#             epsilon=epsilon,
#             annealing_schedule_generator=tight_schedule_annealing,
#             classical_device="fpga",
#             mode=mode,
#             legend_placement=legend_placement,
#             physical_error_rate_min=1e-4,
#             physical_error_rate_max=1e-4,
#             physical_operation_time_min=200e-9,
#             physical_operation_time_max=20_000e-9,
#             physical_measurement_time_min=20e-9,
#             physical_measurement_time_max=2_000e-9,
#             num_trotter_steps=50,
#             statistic=statistic,
#             n_plot_min=3,
#             n_plot_max=100,
#             n_fit_min=5,
#             n_fit_max=10,
#             debug=False,
#             show_schedule_panel=False,
#             show_schedule_vertical_lines=False,
#             legend_y_shift=-0.14,
#             xlabel_labelpad=0,
#         )        
#         plt.savefig(filename + ".png", dpi=300)
#         plt.savefig(filename + ".pgf", dpi=300)
#         print("x", end="")

xx

In [161]:
from monaqa2.data.plot_main23 import plot_annealing_classical_and_quantum_runtime_vs_n, plot_annealing_classical_and_quantum_runtime_fancy_vs_n

beta = 4.0
epsilon = 1e-2
statistic = "mean+std"
MODES = ["compact"]
LEGEND_PLACEMENTS = ["legend_out"]

for mode in MODES:
    for legend_placement in LEGEND_PLACEMENTS:
        filename = f"plots/main_runtime/test/main_classical_quantum_runtime_vs_n_beta{beta:g}_eps{epsilon:.0e}_{statistic.replace('+', '_').replace('/', '_')}_{mode}_{legend_placement}"
        plt.close()
        fig, ax = plot_annealing_classical_and_quantum_runtime_vs_n(
            beta=beta,
            epsilon=epsilon,
            annealing_schedule_generator=tight_schedule_annealing,
            classical_device="fpga",
            mode=mode,
            legend_placement=legend_placement,
            physical_error_rate_min=1e-4,
            physical_error_rate_max=1e-4,
            physical_operation_time_min=200e-9,
            physical_operation_time_max=20_000e-9,
            physical_measurement_time_min=20e-9,
            physical_measurement_time_max=2_000e-9,
            num_trotter_steps=50,
            statistic=statistic,
            n_plot_min=3,
            n_plot_max=100,
            n_fit_min=5,
            n_fit_max=10,
            debug=False,
            show_schedule_panel=False,
            show_schedule_vertical_lines=False,
            legend_y_shift=-0.14,
            xlabel_labelpad=0,
        )        
        plt.savefig(filename + ".png", dpi=300)
        plt.savefig(filename + ".pgf", dpi=300)
        print("x", end="")

x